# 00 · Fase 0 — Encuadre y preparación

**Prueba Técnica Data Scientist · Ferreycorp** — Modelo de Propensión de Compra

Esta fase no analiza ni modela. **Decide y congela** las cuatro cosas de las que dependen
todas las fases siguientes, antes de ajustar nada:

| # | Decisión | Se congela como |
|---|---|---|
| 1 | **Unidad de análisis** — `cliente × visita` | verificada única sobre `(id, dia_visita)` |
| 2 | **Variable objetivo** — `incidencia_compra` | con los universos condicionales de M2 y M3 |
| 3 | **Features admisibles** | tabla con la razón de cada exclusión |
| 4 | **Estrategia de validación** | split temporal en el día **550**, con huella digital |

"Congelar" es literal: al final del notebook, `framing.freeze(df)` escribe el contrato y la
huella del split en `artifacts/encuadre.json`. Las fases posteriores llaman a
`framing.verify_frozen(df)` antes de hacer nada. Si el dato, la frontera o el conjunto de
features cambian en silencio, aparece como una verificación fallada — no como un salto
inexplicable en una métrica.

> **Por qué importa el orden.** Estas decisiones son fáciles de tomar *después* de ver los
> resultados, y ahí dejan de ser decisiones metodológicas y pasan a ser sobreajuste al test.
> Tomarlas ahora, por escrito, es lo que las hace verificables.

La lógica vive en [`src/ferreyros/framing.py`](../src/ferreyros/framing.py).

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from ferreyros import config as cfg, data, framing

pd.set_option("display.max_columns", 40, "display.width", 200, "display.max_colwidth", 95)

df = data.load_raw()
contrato = framing.FramingContract()
print(f"dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas")

dataset: 58,693 filas x 24 columnas


---
## 1 · Unidad de análisis: `cliente × visita`

Cada fila es **una visita de un cliente en un día**. Esa es la unidad sobre la que se predice
y sobre la que el optimizador decide (`x_ijt` se indexa por cliente, marca y período).

La verificación no es formalidad: si `(id, dia_visita)` no identificara una fila única, toda
probabilidad por visita estaría mal planteada y los lags mezclarían filas en silencio.

In [2]:
unidad = framing.verify_unit_of_analysis(df, contrato)
display(unidad)
assert unidad.attrs["unidad_valida"], "La unidad de análisis no está bien definida"

,verificacion,observado,estado
0,id x dia_visita identifica una fila unica,"58,693 pares para 58,693 filas",OK
1,filas en pares duplicados,0,OK
2,cobertura temporal por cliente,72-358 visitas por cliente,OK


**Congelado:** 58,693 pares para 58,693 filas, cero duplicados. La unidad está bien definida.

Importante para lo que viene: cada cliente aparece entre **72 y 358 veces**. El panel es
longitudinal, así que la unidad de *generalización* es el **cliente**, no la fila. De ahí sale
la decisión 4.

---
## 2 · Variable objetivo

El objetivo que pide el enunciado es `incidencia_compra`: *¿comprará o no comprará?*

Pero la función objetivo del problema de optimización necesita tres parámetros
(`π · q · λ`), así que hay tres targets con **universos distintos**. Declarar el universo por
modelo es lo que evita el error más caro disponible aquí: entrenar elección de marca o cantidad
sobre filas donde no hubo compra (control **E12**).

In [3]:
framing.target_definition(df)

,modelo,target,tipo,universo,n_filas,balance,parametro_que_alimenta
0,M1 incidencia,incidencia_compra,clasificacion binaria,todas las visitas,58693,0.2494 positivos,pi_it
1,M2 eleccion de marca,id_marca,clasificacion multiclase (5),solo visitas con compra,14638,1:0.092 / 2:0.310 / 3:0.057 / 4:0.200 / 5:0.340,q_ijt
2,M3 cantidad,cantidad,regresion / conteo,solo visitas con compra,14638,"media 2.77, max 15",lambda_ijt
3,M0 segmentacion,-- (no supervisado),clustering,"500 clientes, demografia",500,n/a,segmento_i


**Congelado:**

- **M1** (el modelo de propensión solicitado) se entrena sobre las **58,693 visitas**, con
  24.94% de positivos.
- **M2** y **M3** se entrenan sobre las **14,638 compras** únicamente. Son modelos
  *condicionales*; usar las 58,693 filas contaminaría el target con no-compras.
- **M0** (segmentación) opera sobre los **500 clientes**, no sobre las visitas.

---
## 3 · Conjunto de features admisibles

Una columna se excluye **aquí y con una razón escrita**, no se filtra después de memoria.
Esta tabla es la que vuelve estructurales los controles E1–E3.

In [4]:
adm = framing.admissibility_table(df)
display(adm)
print(f"admisibles: {adm.attrs['n_admisibles']}  |  "
      f"excluidas: {adm.attrs['n_excluidas']}  |  "
      f"sin clasificar: {adm.attrs['sin_clasificar']}")
assert adm.attrs["sin_clasificar"] == 0, "Hay columnas sin clasificar por el contrato"

,columna,admisible,razon
0,id,NO,Clave de agrupacion; identidad no generaliza a clientes nuevos
1,dia_visita,DERIVADA,No se usa cruda; alimenta features temporales y el split
2,incidencia_compra,NO,Es el target de M1
3,id_marca,NO,"Resultado de la visita, no condicion previa (E2/E3) - fuga"
4,cantidad,NO,"Resultado de la visita, no condicion previa (E2/E3) - fuga"
5,ultima_marca_comprada,SI,Lag verificado contra shift(1) por cliente (E4)
6,ultima_cantidad_comprada,SI,Lag binario del indicador de compra; el diccionario la nombra mal (E5)
7,precio_marca_1,SI,Condicion comercial conocida antes de la visita; decision y_jtk
8,precio_marca_2,SI,Condicion comercial conocida antes de la visita; decision y_jtk
9,precio_marca_3,SI,Condicion comercial conocida antes de la visita; decision y_jtk


admisibles: 18  |  excluidas: 5  |  sin clasificar: 0


**Congelado: 18 columnas crudas admisibles, 5 excluidas.** Las exclusiones, por razón:

| Columna | Por qué no entra |
|---|---|
| `tamanio_ciudad` | Indicación explícita del diccionario (**E1**) |
| `incidencia_compra` | Es el target de M1 |
| `id_marca`, `cantidad` | **Resultados** de la visita, no condiciones previas (**E2/E3**) — serían fuga |
| `id` | La identidad del cliente no generaliza a clientes nuevos |

`dia_visita` es un caso aparte: **no se usa cruda** (el día 600 no significa nada para un
cliente futuro), pero alimenta las features temporales y define el split.

Las dos columnas `ultima_*` **sí** entran, y entran con el significado que se verificó en el
EDA, no con el que dice el diccionario: `ultima_marca_comprada` es un lag real (coincidencia
1.0 con `shift(1)` por cliente) y `ultima_cantidad_comprada` es **binaria** — el lag del
indicador de compra, no una cantidad.

In [5]:
# Lo que nunca puede aparecer en la matriz de M1, congelado como conjunto
sorted(framing.forbidden_columns())

['cantidad',
 'gasto',
 'id',
 'id_marca',
 'incidencia_compra',
 'precio_pagado',
 'tamanio_ciudad']

Nótese que `gasto` y `precio_pagado` también están prohibidas: son columnas derivadas que el
EDA construye para medir ingreso, pero dependen de la marca elegida y de la cantidad — es
decir, son resultados de la visita igual que sus fuentes.

---
## 4 · Estrategia de validación

Cada decisión aquí está **forzada por la estructura del dataset**, no elegida por preferencia.
Esa es la columna `razon`.

In [6]:
framing.validation_plan(contrato)

,aspecto,decision,razon
0,Tipo de split,temporal (no aleatorio),"Con ~117 visitas por cliente, un split aleatorio reparte visitas del mismo cliente entre tr..."
1,Frontera del split,train: dia <= 550 | test: dia > 550,Deja ~180 dias de test (25%) conservando los 500 clientes y la tasa base.
2,Validacion cruzada en train,TimeSeriesSplit en train + GroupKFold por id para clientes nuevos,TimeSeriesSplit respeta el orden temporal para el tuning; GroupKFold por id mide generaliza...
3,Metrica principal,PR-AUC,"Desbalance 25/75: PR-AUC es sensible a la clase positiva, que es la que importa."
4,Metrica prohibida,accuracy (75.1% prediciendo siempre no-compra),Predecir siempre 'no compra' da 75.1% de accuracy y cero valor de negocio.
5,Umbral de decision,"derivado del costo de contacto en P1, no fijado en 0.5","El umbral optimo iguala margen esperado y costo de contacto: lo fija la optimizacion P1, no..."
6,Baselines a superar,tasa base 0.2494 | targeting historico lift x2.43,Un modelo que no bate el ordenamiento por tasa historica no justifica su existencia.
7,Preprocesamiento,Pipeline de scikit-learn con fit solo sobre train (E10),Ajustar escalado o codificacion sobre todo el dataset filtra informacion del test.


### 4.1 · Congelar el split temporal

Frontera en el día **550**: train ≈ días 1–550, test ≈ días 551–730.

La huella digital (`fingerprint`) es un hash de los pares `(id, dia_visita)` de cada lado.
Cambia si cambia el dato o si se mueve la frontera, que es exactamente lo que queremos detectar.

In [7]:
split = framing.split_manifest_table(df, contrato)
display(split)
print("split balanceado:", split.attrs["balanceado"])

,split,filas,clientes,dia_min,dia_max,tasa_compra,fingerprint
0,train,45497,500,1,550,0.2498,47959243b182b9af
1,test,13196,500,551,730,0.2480,842f57dd85e58abc


split balanceado: True


**Congelado:**

| | filas | clientes | días | tasa de compra |
|---|---|---|---|---|
| train | 45,497 | 500 | 1–550 | 0.2498 |
| test | 13,196 | 500 | 551–730 | 0.2480 |

Ambos lados conservan **los 500 clientes** y prácticamente la misma tasa base (0.2498 vs
0.2480), así que el corte no introduce sesgo de composición (control **E9**).

Un `train_test_split` aleatorio sería **inválido**: con ~117 visitas por cliente repartiría
visitas del mismo cliente entre train y test e inflaría todas las métricas de forma
irreparable. El test mide algo distinto y más honesto: *predecir el futuro de clientes
conocidos*. Para medir generalización a **clientes nuevos** —el riesgo real con solo 500
clientes— se usa en paralelo `GroupKFold` por `id`.

---
## 5 · Congelar el encuadre

In [8]:
reporte = framing.phase0_report(df, contrato)
display(reporte)
assert reporte.attrs["fase0_completa"], "La Fase 0 tiene decisiones sin cerrar"

,decision,definicion,estado
0,1. Unidad de analisis,cliente x visita (id x dia_visita),OK
1,2. Variable objetivo,incidencia_compra (M1); id_marca/cantidad condicionados a compra,OK
2,3. Features admisibles,"18 columnas crudas admisibles, 5 excluidas con razon",OK
3,4. Estrategia de validacion,"temporal (no aleatorio), corte dia 550; PR-AUC",OK


In [9]:
payload = framing.freeze(df, contrato)
print(f"contrato escrito en: {framing.FRAMING_PATH.relative_to(ROOT)}")
print()
print(json.dumps(payload["split"], indent=2))

contrato escrito en: artifacts/encuadre.json

{
  "split_day": 550,
  "train": {
    "filas": 45497,
    "clientes": 500,
    "dia_min": 1,
    "dia_max": 550,
    "tasa_compra": 0.2498,
    "fingerprint": "47959243b182b9af"
  },
  "test": {
    "filas": 13196,
    "clientes": 500,
    "dia_min": 551,
    "dia_max": 730,
    "tasa_compra": 0.248,
    "fingerprint": "842f57dd85e58abc"
  },
  "balanceado": true
}


In [10]:
display(contrato.as_table())

,decision,valor
0,unidad_analisis,cliente x visita
1,clave_unidad,"id, dia_visita"
2,target_m1,incidencia_compra
3,target_m2,id_marca
4,target_m3,cantidad
5,universo_m1,"todas las visitas (58,693)"
6,universo_m2_m3,"visitas con incidencia_compra == 1 (14,638)"
7,split_day,550
8,split_tipo,temporal (no aleatorio)
9,cv_estrategia,TimeSeriesSplit en train + GroupKFold por id para clientes nuevos


### 5.1 · Verificar el congelado

`verify_frozen` es lo que se ejecuta al inicio de cada fase posterior.

In [11]:
ver = framing.verify_frozen(df)
display(ver)
assert ver.attrs["coincide"], "El dataset ya no coincide con el encuadre congelado"

,verificacion,congelado,actual,estado
0,frontera del split,550,550,OK
1,filas totales,58693,58693,OK
2,clientes,500,500,OK
3,fingerprint train,47959243b182b9af,47959243b182b9af,OK
4,fingerprint test,842f57dd85e58abc,842f57dd85e58abc,OK


---
## 6 · Cierre del círculo: la Fase 2 obedece el contrato

La Fase 0 congela; la Fase 2 (`preprocess.prepare()`) construye las matrices. Que el vínculo
sea **mecánico y no una cuestión de confianza** es el punto de esta última verificación: se
comprueba que la frontera, los conteos de filas, la partición de clientes y la admisibilidad de
las features son los que se congelaron.

In [12]:
from ferreyros import preprocess

prepared = preprocess.prepare()
display(prepared.shapes)

,conjunto,filas,cols_entrada,cols_salida,tasa_compra,clientes
0,train,45497,60,83,0.2498,500
1,test,13196,60,83,0.2480,500


In [13]:
obedece = framing.verify_prepared_against_contract(prepared)
display(obedece)
assert obedece.attrs["obedece_contrato"], "Las matrices de modelado no respetan el encuadre"
print("Las matrices de modelado respetan el encuadre congelado.")

,verificacion,congelado,actual,estado
0,frontera del split,550,550,OK
1,filas de train,45497,45497,OK
2,filas de test,13196,13196,OK
3,tasa de compra en train,0.2498,0.2498,OK
4,tasa de compra en test,0.248,0.248,OK
5,clientes presentes en ambos lados,500,500,OK
6,ninguna columna prohibida en la matriz,7 prohibidas,ninguna,OK


Las matrices de modelado respetan el encuadre congelado.


**Resultado:** las 7 verificaciones pasan. `prepare()` produjo 45,497 filas de train y 13,196
de test, con las tasas congeladas, los 500 clientes en ambos lados y ninguna columna prohibida
entre las 60 features de entrada.

---
## 7 · Síntesis de la Fase 0

| Decisión | Congelado |
|---|---|
| Unidad de análisis | `cliente × visita`, única sobre `(id, dia_visita)` — 58,693 filas |
| Target M1 | `incidencia_compra`, 24.94% positivos, universo completo |
| Targets M2 / M3 | `id_marca` / `cantidad`, universo condicional de 14,638 compras |
| Features admisibles | 18 columnas crudas; 5 excluidas con razón; 7 prohibidas por contrato |
| Split | temporal, día 550 · train 45,497 @ 0.2498 · test 13,196 @ 0.2480 |
| Huellas digitales | train `47959243b182b9af` · test `842f57dd85e58abc` |
| Validación cruzada | `TimeSeriesSplit` en train + `GroupKFold` por `id` |
| Métrica principal | PR-AUC (accuracy explícitamente prohibida) |
| Umbral de decisión | derivado del costo de contacto en P1, no 0.5 |
| Baselines a superar | tasa base 0.2494 · targeting histórico lift ×2.43 |

### Lo que esta fase deja cerrado

1. **Nada de lo anterior se puede renegociar después de ver métricas.** El contrato está en
   `artifacts/encuadre.json` con huella digital; moverlo exige reescribirlo y decirlo.
2. **La fuga de datos es estructural, no disciplinaria.** `framing.forbidden_columns()` es la
   única fuente de verdad, y `feature_matrix()` la respeta por construcción.
3. **El split está congelado antes de ajustar nada.** No hay forma de elegir la frontera que
   mejor le quede al modelo.

### Siguiente paso

`01_eda.ipynb` ya corre sobre este encuadre (bloques A–D de la matriz de alineamiento), y
`02_modelado.ipynb` arranca llamando a `framing.verify_frozen(df)` antes de entrenar M0/M1/M2/M3.